# Vector stores and semantic search



In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer

c:\Users\kevin\miniconda3\envs\deeplearning\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Part I: Basic vector store implementation

In [2]:
class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document


class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self._model = embedding_model
        self._documents: list[Document] = []
        self._embeddings: np.ndarray | None = None

    def add_documents(self, documents: list[Document]):
        texts = [doc.text for doc in documents]
        new_embeddings = self._model.encode(texts, convert_to_numpy=True, show_progress_bar=True)
        # Normalizamos
        norms = np.linalg.norm(new_embeddings, axis=1, keepdims=True)
        new_embeddings = new_embeddings / np.maximum(norms, 1e-10)

        self._documents.extend(documents)
        if self._embeddings is None:
            self._embeddings = new_embeddings
        else:
            self._embeddings = np.vstack([self._embeddings, new_embeddings])

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        if self._embeddings is None or len(self._documents) == 0:
            return []

        query_emb = self._model.encode([query], convert_to_numpy=True)
        query_emb = query_emb / np.maximum(np.linalg.norm(query_emb, keepdims=True), 1e-10)

        scores = (self._embeddings @ query_emb.T).flatten()

        top_indices = np.argsort(scores)[::-1][:top_k]
        return [
            SearchResult(score=float(scores[i]), document=self._documents[i])
            for i in top_indices
        ]

## Cargar el dataset

In [5]:
import pandas as pd

DATASET_URL = "https://raw.githubusercontent.com/ekohrt/animal-fun-facts-dataset/main/animal-fun-facts-dataset.csv"

df = pd.read_csv(DATASET_URL)
print(df.shape)
df.head()

(7734, 5)


,animal_name,source,text,media_link,wikipedia_link
0,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,"Aardvarks are sometimes called ""ant bears"", ""e...",NaN,/wiki/Aardvark
1,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Aardvarks\nhave rather primitive brains that a...,NaN,/wiki/Aardvark
2,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Aardvarks\nteeth are lined with fine upright t...,NaN,/wiki/Aardvark
3,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,"The aardvarks Latin family name ""Tubulidentata...",NaN,/wiki/Aardvark
4,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Baby aardvarks are born with front teeth that ...,NaN,/wiki/Aardvark


In [9]:
# Aquí creamos los documentos
documents = [
    Document(
        text=row["text"],
        metadata={
            "animal_name":     str(row.get("animal_name", "")),
            "source":          str(row.get("source", "")),
            "media_link":      str(row.get("media_link", "")),
            "wikipedia_link":  str(row.get("wikipedia_link", "")),
        },
    )
    for _, row in df.iterrows()
    if pd.notna(row.get("text")) and str(row.get("text", "")).strip()
]

## Indexing y consultas

In [10]:
model = SentenceTransformer("all-MiniLM-L6-v2")

vs = VectorStore(embedding_model=model)
vs.add_documents(documents)

print(f"VectorStore listo con {len(vs._documents)} documentos.")

Batches: 100%|██████████| 242/242 [00:02<00:00, 102.24it/s]

VectorStore listo con 7731 documentos.


In [11]:
def print_results(query: str, results: list[SearchResult]):
    print(f"Query: '{query}'")
    print("-" * 70)
    for i, r in enumerate(results, 1):
        print(f"  #{i}  score={r.score:.4f}")
        print(f"       text    : {r.document.text[:120]}...")
        print(f"       metadata: {r.document.metadata}")
        print()

queries = [
    "animals that live in the ocean",
    "fastest animals on land",
    "animals with unique sleeping habits",
    "animals that use tools",
    "venomous or poisonous creatures",
]

for q in queries:
    results = vs.search(q, top_k=3)
    print_results(q, results)
    print("=" * 70)
    print()

Query: 'animals that live in the ocean'
----------------------------------------------------------------------
  #1  score=0.6498
       text    : Smallest cetacean in the ocean...
       metadata: {'animal_name': 'vaquita', 'source': 'https://a-z-animals.com/animals/vaquita/', 'media_link': 'nan', 'wikipedia_link': '/wiki/Vaquita'}

  #2  score=0.6452
       text    : White Sharks live in all of the world's oceans....
       metadata: {'animal_name': 'white shark', 'source': 'https://a-z-animals.com/animals/white-shark/', 'media_link': 'nan', 'wikipedia_link': '/wiki/Great_white_shark'}

  #3  score=0.6317
       text    : May eat squid or other small invertebrate ocean life...
       metadata: {'animal_name': 'bonito fish', 'source': 'https://a-z-animals.com/animals/bonito-fish/', 'media_link': 'nan', 'wikipedia_link': '/wiki/Bonito'}


Query: 'fastest animals on land'
----------------------------------------------------------------------
  #1  score=0.8567
       text    : Fastest a

## Part II: Filtering by metadata

In [7]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        pass

    def add_documents(self, documents: list[Document]):
        pass

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        pass